In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from utils import *
from reprojection import *
import pandas as pd

In [2]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers.csv', skipinitialspace=True).dropna()
xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()
dids = df['did'].to_numpy()

s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xd, yd = s['xd'], s['yd']


def fit(data, header, did, mu_thr=0.1):
    data = undistort(data, header, xd, yd)

    view = View.from_header(header)
    view.update(crota=view.crota + 0.29, xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], rsun=ru_sun[dids == did][0] + 0.28)

    view_ = View.from_header(header)
    view_.update(crota=view.crota + 0.29 + 180, xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], rsun=ru_sun[dids == did][0] + 0.28)

    V = (view.velocity(mu_thr=mu_thr, cbs=False) - view_.velocity(mu_thr=mu_thr, cbs=False)) / 2
    V_ = (data - view_.reproject(data, view)) / 2 * 1000

    t = ~np.isnan(V)
    x, y = V[t], V_[t]

    k = np.nanmean(x * y) / np.nanmean(x ** 2)

    V = view.velocity(mu_thr=mu_thr, cbs=False)
    V -= np.nanmean(V)

    V_ = data * 1000
    V_[np.isnan(V)] = np.nan
    V_ -= np.nanmean(V_)

    U = V_ - V * k

    mu = view.mu(thr=mu_thr)

    t = np.where(~np.isnan(mu))
    p = np.polyfit(mu[t], U[t], 3)
    return k, p

In [3]:
import fnmatch
from connect import bob

np.set_printoptions(suppress=True)
sftp = bob()

top_dir = '/data/slam/valori/test_l2_fmdb/FDT_test_release_v08_2025/v4/l2/'
dirs = sorted(sftp.listdir(top_dir))

output_file = 'doppler_velocities_.csv'
with open(output_file, 'w') as f:
    f.write('date, did, dsun_au, obs_vr, fgov1pt1, k, p0, p1, p2, p3\n')


In [ ]:
for directory in dirs:
    if directory[:4] in ['2024', '2025']:
        for file in sorted(sftp.listdir(top_dir + directory)):

            if fnmatch.fnmatch(file, '*fdt-vlos*.fits.gz'):
                try:
                    remote_file = top_dir + directory + '/' + file
                    local_file = 'temp.fits.gz'
                    
                    sftp.get(remote_file, local_file)
                    
                    with fits.open(local_file) as hdul:
                        data = hdul[0].data
                        header = hdul[0].header

                    date = file.split('_')[3]
                    did = file.split('_')[-1].split('.')[0]
                    dsun_au = header['DSUN_AU']
                    obs_vr = header['OBS_VR']
                    fgov1pt1 = header['FGOV1PT1']

                    k, p = fit(data, header, int(did))

                    with open(output_file, 'a') as f:
                        f.write(f'{date}, {did}, {dsun_au:.4f}, {obs_vr:.4f}, {fgov1pt1}, {k:.4f}, {p[0]:.4f}, {p[1]:.4f}, {p[2]:.4f}, {p[3]:.4f}\n')
                        
                except:
                    print(file)
                    continue

sftp.close()